# SDF Viewer (SdfVisualizer)

**Part I · Visualization** — Tutorial 21

Introduce the dedicated, fullscreen ray-marched viewer
`pytanga.viz.sdf.SdfVisualizer`. You will learn to:

- Start the viewer, add entities, and render SDF primitives.
- Bundle constituents with `Composed` and apply per-object CSG.
- Use the automatic entity→SDF mapping and configurable lighting.
- Drive the simple update loop (`update_entity` / `update_light` / `flush` /
  `sleep_ms`).

> **Note:** this is an early-stage, experimental rendering path (WebGL2 only,
> 3D only). The supported, feature-rich route for SDF objects is
> [Tutorial 05](../05_sdf_objects/).


## Setup


In [ ]:
from pytanga.geometry import Direction, Line, Point, Rotor, Sphere
from pytanga.viz.sdf import SdfVisualizer


## 1. Quick start

`SdfVisualizer` + `add()` + `show()` / `wait()` — the same ergonomics as the
standard viewer, but every object is ray-marched.


In [ ]:
viz = SdfVisualizer(title="SDF — quick start")
viz.add(Sphere(Point(0, 0, 0), 1.0), color="#ffaa00")

viz.show()  # opens the SDF viewer in a browser
# viz.wait()  # blocks until Ctrl+C
print("SDF viewer shown")


## 2. The SDF primitive library

Primitives are exposed directly as `SdfNode` objects:
`sphere`, `box`, `cylinder`, `capped_cylinder`, `cone`, `capped_cone`, `torus`,
`ellipsoid`, `round_box`, `capsule`, `segment`, `plane`, `bound_box`.


In [ ]:
from pytanga.viz.sdf import box, capped_cylinder, sphere, torus

viz = SdfVisualizer(title="SDF — primitives")

viz.add(sphere(0.7, position=(0, 0, 0)), color="#ffaa00")
viz.add(box((0.9, 0.9, 0.9), position=(2.0, 0, 0)), color="#4499ff")
viz.add(torus(1.1, 0.12, position=(-2.0, 0, 0)), color="#44ff44")
viz.add(capped_cylinder(1.0, 0.45, position=(0, 2, 0)), color="#ff44ff")

# viz.show(); viz.wait()
print("primitives added")


## 3. `Composed` — per-constituent combine modes

`Composed` bundles constituents (each with its own combine mode) into one
object with one material. Use the `(obj, "subtract")` tuple form for a
constituent's combine mode.


In [ ]:
from pytanga.viz.sdf import Composed, capped_cylinder, sphere

viz = SdfVisualizer(title="SDF — Composed")

# A bead: a sphere with a cylinder bored through it.
bead = Composed(
    sphere(0.7, position=(0.0, 0.0, 0.0)),
    (capped_cylinder(1.0, 0.45, position=(0.0, 0.0, 0.0)), "subtract"),
)
viz.add(bead, color="#ffaa00")

# viz.show(); viz.wait()
print("Composed bead added")


## 4. Per-object CSG — `combine=` / `polarity=`

`combine=` (or the `polarity=` shorthand) folds an object into the composed SDF:
`"union"` / `"intersection"` / `"subtract"` (and `"smooth_union"` /
`"smooth_intersection"` / `"smooth_subtract"` with a `smoothness=` knob).


In [ ]:
viz = SdfVisualizer(title="SDF — booleans")

# Union (default):
viz.add(Sphere(Point(0, 0, 0), 1.6), color="#ffaa00")

# Subtract — carve a cavity:
viz.add(Sphere(Point(0.9, 0.5, 0), 0.8), combine="subtract")

# Intersection — keep only the overlap:
viz.add(Sphere(Point(-1.1, 0, 0), 1.2), color="#44aaff", combine="intersection")

# viz.show(); viz.wait()
print("boolean combine modes added")


## 5. Automatic entity → SDF mapping

Geometry entities and operators are mapped to SDF primitives automatically:
`Point`→sphere, `Line`→segment, `Sphere`→sphere, `Rotor`→disc+ring+axis, etc.


In [ ]:
viz = SdfVisualizer(title="SDF — entity mapping")

viz.add(Point(0, 0, 0), color="#ffaa00")                                   # → sphere
viz.add(Line.from_points(Point(-3, 0, 0), Point(3, 0, 0)), color="#44ff44", thickness=0.08)  # → segment
viz.add(Sphere(Point(0, 1.5, 0), 1.0), color="#ff44ff")                    # → sphere
viz.add(Rotor(angle=1.0, axis=Direction(0, 0, 1)), color="#4488ff")        # → disc+ring+axis

# viz.show(); viz.wait()
print("entities mapped to SDF")


## 6. Lighting

`DirectionalLight` adds a directional light; `add_default_light=False` disables
the default light; `set_ambient_light(color=..., intensity=...)` tunes the
ambient term.


In [ ]:
from pytanga.viz.sdf import DirectionalLight

viz = SdfVisualizer(title="SDF — lighting", add_default_light=False)
viz.add(Sphere(Point(0, 0, 0), 1.5), color="#ffaa00")

key = DirectionalLight(direction=(4.0, 0.0, 3.0), color="#ffffff", intensity=1.2)
viz.add(key)
viz.add(DirectionalLight(direction=(-2.0, -1.0, 1.0), color="#8899ff", intensity=0.35))
viz.set_ambient_light(color="#ffffff", intensity=0.3)

# viz.show(); viz.wait()


## 7. Grid / axes overlays and the update loop

`SdfVisualizer(add_default_grid=..., add_default_axes=...)` draws shader-based
`Grid` / `Axes` overlays. The simple update loop is `update_entity()` /
`update_light()` / `flush()` / `sleep_ms()`.


In [ ]:
viz = SdfVisualizer(title="SDF — update loop", add_default_grid=True, add_default_axes=True)

sphere_id = viz.add(Sphere(Point(0, 0, 0), 1.5), color="#ffaa00")

# Move the sphere each frame (simple loop — no animate()):
# for step in range(120):
#     x = math.sin(step * 0.05) * 2.0
#     viz.update_entity(sphere_id, Sphere(Point(x, 0, 0), 1.5))
#     viz.flush()
#     if not viz.sleep_ms(16):   # False == interrupted
#         break

print("update loop covered")


## Visual Examples

The library's SDF demos in condensed form — line + sphere − sphere, booleans,
`Composed`, and a moving light.


In [ ]:
import math

viz = SdfVisualizer(title="SDF — entities (line + sphere − sphere)", add_default_light=False)

# Line segment:
viz.add(Line.from_points(Point(-3, 0, 0), Point(3, 0, 0)), color="#44ff44", thickness=0.08)

# Filled sphere:
viz.add(Sphere(Point(0, 1.5, 0), radius=1.5), color="#ffaa00")

# A subtracting sphere carves a bite out of it:
viz.add(Sphere(Point(1.2, 2.0, 0), radius=0.9), combine="subtract")

# Moving key light:
key = DirectionalLight(direction=(4.0, 0.0, 3.0), color="#ffffff", intensity=1.2)
viz.add(key)
viz.set_ambient_light(color="#ffffff", intensity=0.3)

# while True:
#     t += 0.03
#     key.direction = (math.cos(t) * 4.0, math.sin(t) * 4.0, 3.0)
#     viz.flush()
#     if not viz.sleep_ms(16):
#         break

# viz.show(); viz.wait()
print("SDF viewer scene built")


## Summary

| Task | API |
|---|---|
| Viewer | `SdfVisualizer()` + `add()` + `show()` / `wait()` |
| Primitives | `sphere`, `box`, `cylinder`, `capped_cylinder`, `cone`, `capped_cone`, `torus`, `ellipsoid`, `round_box`, `capsule`, `segment`, `plane`, `bound_box` |
| Bundle | `Composed(part, (part, "subtract"), …)` |
| Per-object CSG | `add(obj, combine="union" / "intersection" / "subtract", smoothness=…)` |
| Entity mapping | `Point`→sphere, `Line`→segment, `Rotor`→disc+ring+axis, … |
| Lighting | `DirectionalLight` / `add_default_light` / `set_ambient_light` |
| Overlays | `add_default_grid` / `add_default_axes` |
| Update loop | `update_entity` / `update_light` / `flush` / `sleep_ms` |

This completes Part I — Visualization.
